In [1]:
# ============================
# RAG Demo using FAISS + FLAN-T5
# Single Cell Google Colab Code
# ============================

# Install required libraries
!pip -q install sentence-transformers transformers faiss-cpu torch

import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading embedding model...")

# -------------------------------
# Knowledge Base
# -------------------------------

documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation (RAG) combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in Artificial Intelligence.",
    "Vector databases store embeddings and support fast similarity search.",
    "FAISS is a library developed by Meta for efficient similarity search.",
    "Sentence Transformers convert text into dense vector embeddings."
]

# -------------------------------
# Load Sentence Transformer
# -------------------------------

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")

# -------------------------------
# Create Embeddings
# -------------------------------

doc_embeddings = embed_model.encode(documents)

# -------------------------------
# Build FAISS Index
# -------------------------------

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(doc_embeddings))

print("FAISS Index Created.")

# -------------------------------
# User Query
# -------------------------------

query = "What is Retrieval-Augmented Generation in AI?"

query_embedding = embed_model.encode([query])

# -------------------------------
# Retrieve Top-2 Documents
# -------------------------------

k = 2

distances, indices = index.search(np.array(query_embedding), k)

retrieved_docs = [documents[i] for i in indices[0]]

print("\nRetrieved Documents:\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i}. {doc}")

# -------------------------------
# Build Prompt
# -------------------------------

context = " ".join(retrieved_docs)

prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

# -------------------------------
# Load FLAN-T5
# -------------------------------

print("\nLoading FLAN-T5 model...")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("FLAN-T5 Loaded Successfully.")

# -------------------------------
# Generate Answer
# -------------------------------

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=80
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

# -------------------------------
# Final Output
# -------------------------------

print("\n==============================")
print("QUESTION")
print("==============================")
print(query)

print("\n==============================")
print("RETRIEVED CONTEXT")
print("==============================")

for doc in retrieved_docs:
    print("-", doc)

print("\n==============================")
print("GENERATED ANSWER")
print("==============================")
print(answer)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 30.4 MB/s eta 0:00:00
Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 Loaded Successfully.

QUESTION
What is Retrieval-Augmented Generation in AI?

RETRIEVED CONTEXT
- Retrieval-Augmented Generation (RAG) combines document retrieval with text generation.
- Vector databases store embeddings and support fast similarity search.

GENERATED ANSWER
combines document retrieval with text generation
